# Interpolate data onto an even time step
Author: Brian Green (briangre@stanford.edu)

In [9]:
import pandas as pd
import numpy as np
import warnings
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import netCDF4 as nc
from scipy import interpolate

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    tqdm.pandas()

%matplotlib inline

#INTERP_DT = '30S' # the time step to interpolate the data to
#INTERP_DT_NP = np.timedelta64(30,'s') # the same as above, but in a format that numpy can use
INTERP_DT = '120S' # the time step to interpolate the data to
INTERP_DT_NP = np.timedelta64(120,'s') # the same as above, but in a format that numpy can use

In [10]:
# Load in the data

# Data that has already been split into segments across data gaps:
flights = pd.read_feather('Concordiasi_with_ERA5_4_spikes.feather')

## Data that haven't been split into data gaps:
#flights = pd.read_feather('Concordiasi_with_ERA5_2_takeoffs_landings.feather')
## Give each flight a segment_id, to be consistent with Loon and Concordiasi
#flight_ids = flights.flight_id.unique()
#n_flights = flights.flight_id.nunique()
#for i in np.arange(n_flights):
#    flights.loc[flights['flight_id'] == flight_ids[i],['segment_id']] = i+1
#flights = flights.astype({'segment_id': int})

flights

,flight_id,time,latitude,longitude,altitude,u,v,pressure,T_ERA5,N2_ERA5,superpressure,segment_id
0,01,2010-09-23 01:35:00,-78.406868,168.230804,17418.7,8.44,-9.58,61.573,192.615719,0.000580,7.54,0
1,01,2010-09-23 01:35:30,-78.409485,168.242310,17415.3,8.36,-9.75,61.616,192.613123,0.000580,7.70,0
2,01,2010-09-23 01:36:00,-78.412132,168.253815,17408.5,8.37,-9.82,61.715,192.607186,0.000579,7.83,0
3,01,2010-09-23 01:36:30,-78.414772,168.265289,17403.8,8.40,-9.79,61.751,192.605042,0.000579,7.97,0
4,01,2010-09-23 01:37:00,-78.417358,168.276642,17405.5,8.43,-9.68,61.691,192.608620,0.000579,8.11,0
...,...,...,...,...,...,...,...,...,...,...,...,...
2379058,19,2010-11-23 09:56:00,-76.331482,-20.582638,17357.0,10.34,6.53,66.633,207.918384,0.000660,11.57,167
2379059,19,2010-11-23 09:56:30,-76.329704,-20.570866,17358.1,10.33,6.50,66.584,207.932783,0.000660,11.60,167
2379060,19,2010-11-23 09:57:00,-76.327957,-20.559189,17359.9,10.30,6.47,66.571,207.936609,0.000660,11.62,167
2379061,19,2010-11-23 09:57:30,-76.326241,-20.547585,17361.2,10.28,6.44,66.583,207.933077,0.000660,11.63,167


In [11]:
def interp_segment(segment):
    
    # Make the data numpy arrays
    time = np.array(segment.time)
    latitude = np.array(segment.latitude)
    longitude = np.array(segment.longitude)
    wind_u = np.array(segment.u)
    wind_v = np.array(segment.v)
    altitude = np.array(segment.altitude)
    pressure = np.array(segment.pressure)
    T_ERA5 = np.array(segment.T_ERA5)
    N2_ERA5 = np.array(segment.N2_ERA5)
    superpressure = np.array(segment.superpressure)
    
    # Set the time range to prevent extrapolation if there are NaNs in the wind or altitude data
    temp = wind_u + wind_v + altitude
    temp_time = time[np.isnan(temp) == 0]
    
    # Create the new time series and the dataframe
    # Set the start time so that it's the first multiple of INTERP_DT_NP after the start
    # of the segment. This ensures all the segments are on the same time grid.
    ref_time = np.datetime64('2000-01-01T00:00:00.000000000')
    start_time = np.ceil((time[0] - ref_time)/INTERP_DT_NP)
    start_time = ref_time + INTERP_DT_NP*start_time
    new_time = pd.date_range(start_time, temp_time[-1], freq=INTERP_DT)
    interpd = pd.DataFrame(columns=['flight_id','time','latitude','longitude','altitude',
                                    'pressure','wind_u','wind_v','T_ERA5','N2_ERA5',
                                    'superpressure','segment_id'])
    interpd.time = new_time
    interpd.flight_id = segment.flight_id.iloc[0]
    interpd.segment_id = segment.segment_id.iloc[0]
    
    # Times must be converted to a float before fields can be interpolated
    time_float = np.array((time - time[0])/pd.Timedelta(1, 'minutes'))
    new_time_float = np.array((new_time - time[0])/pd.Timedelta(1, 'minutes'))
    N_temp = np.size(new_time_float)
    
    # There can be NaNs in some fields. Delete these and make new time series for interpolation
    time_float_lat = time_float[np.isnan(latitude) == 0]
    time_float_lon = time_float[np.isnan(longitude) == 0]
    time_float_alt = time_float[np.isnan(altitude) == 0]
    time_float_u = time_float[np.isnan(wind_u) == 0]
    time_float_v = time_float[np.isnan(wind_v) == 0]
    time_float_p = time_float[np.isnan(pressure) == 0]
    time_float_T = time_float[np.isnan(T_ERA5) == 0]
    time_float_N2 = time_float[np.isnan(N2_ERA5) == 0]
    time_float_SP = time_float[np.isnan(superpressure) == 0]
    latitude = latitude[np.isnan(latitude) == 0]
    longitude = longitude[np.isnan(longitude) == 0]
    altitude = altitude[np.isnan(altitude) == 0]
    wind_u = wind_u[np.isnan(wind_u) == 0]
    wind_v = wind_v[np.isnan(wind_v) == 0]
    pressure = pressure[np.isnan(pressure) == 0]
    T_ERA5 = T_ERA5[np.isnan(T_ERA5) == 0]
    N2_ERA5 = N2_ERA5[np.isnan(N2_ERA5) == 0]
    superpressure = superpressure[np.isnan(superpressure) == 0]
    # For dealing with the date line:
    temp_longitude = longitude.copy()
    temp_longitude[temp_longitude < 0] = temp_longitude[temp_longitude < 0] + 360
    
    # Do the interpolation
    f1 = interpolate.PchipInterpolator(time_float_lat, latitude)
    f2 = interpolate.PchipInterpolator(time_float_lon, longitude)
    f2b = interpolate.PchipInterpolator(time_float_lon, temp_longitude)
    f3 = interpolate.PchipInterpolator(time_float_alt, altitude)
    f4 = interpolate.PchipInterpolator(time_float_p, pressure)
    f5 = interpolate.PchipInterpolator(time_float_u, wind_u)
    f6 = interpolate.PchipInterpolator(time_float_v, wind_v)
    if T_ERA5.size > 0: # one flight doesn't have any ERA5 data
        f7 = interpolate.PchipInterpolator(time_float_T, T_ERA5)
        f8 = interpolate.PchipInterpolator(time_float_N2, N2_ERA5)
    f9 = interpolate.PchipInterpolator(time_float_SP, superpressure)
    interpd.latitude = f1(new_time_float)
    longitude1 = f2(new_time_float)
    longitude2 = f2b(new_time_float)
    interpd.altitude = f3(new_time_float)
    interpd.pressure = f4(new_time_float)
    interpd.wind_u = f5(new_time_float)
    interpd.wind_v = f6(new_time_float)
    if T_ERA5.size > 0: # one flight doesn't have any ERA5 data
        interpd.T_ERA5 = f7(new_time_float)
        interpd.N2_ERA5 = f8(new_time_float)
    else:
        interpd.T_ERA5 = np.nan
        interpd.N2_ERA5 = np.nan
    interpd.superpressure = f9(new_time_float)
    
    ## Deal with balloons crossing the date line
    # Identify intervals where the balloon crossed the date line
    dlon = np.array(segment.longitude)
    dlon = np.diff(dlon)
    indices = 1 + np.arange(np.size(dlon))
    indices_new = np.arange(np.size(new_time_float))
    if np.nanmax(np.abs(dlon)) < 180:
        interpd.longitude = longitude1
    else:
        ind_crossings = indices[np.abs(dlon) > 180]
        n_crossings = np.size(ind_crossings)
        for i in np.arange(n_crossings):
            ind_start = ind_crossings[i] - 1
            ind_stop = ind_crossings[i]
            time_float_start = time_float[ind_start]
            time_float_stop = time_float[ind_stop]
            # Find new_time_float indices in this time range
            temp_time = new_time_float.copy()
            ind_new_crossing = indices_new[temp_time > time_float_start]
            temp_time = temp_time[temp_time > time_float_start]
            ind_new_crossing = ind_new_crossing[temp_time < time_float_stop]
            # Replace longitude1 with longitude2 for those indices
            longitude1[ind_new_crossing] = longitude2[ind_new_crossing]
        # Make sure the longitude range is -180 to 180
        longitude1[longitude1 > 180] = longitude1[longitude1 > 180] - 360
        interpd.longitude = longitude1
    
    return interpd

flights_interpolated = flights.groupby('segment_id').progress_apply(interp_segment).reset_index(drop=True)

  0%|          | 0/108 [00:00<?, ?it/s]

/opt/anaconda3/envs/LOON/lib/python3.9/site-packages/scipy/interpolate/_cubic.py:289: RuntimeWarning: invalid value encountered in add
  whmean = (w1/mk[:-1] + w2/mk[1:]) / (w1 + w2)


In [12]:
# Save the data
flights_interpolated.to_feather('Concordiasi_with_ERA5_5_interpolated_2minDT.feather') 
flights_interpolated

,flight_id,time,latitude,longitude,altitude,pressure,wind_u,wind_v,T_ERA5,N2_ERA5,superpressure,segment_id
0,01,2010-09-23 01:36:00,-78.412132,168.253815,17408.5,61.715,8.37,-9.82,192.607186,0.000579,7.83,0
1,01,2010-09-23 01:38:00,-78.422356,168.298798,17423.1,61.571,8.33,-9.44,192.615840,0.000580,8.39,0
2,01,2010-09-23 01:40:00,-78.432671,168.341537,17411.3,61.658,7.75,-9.70,192.610597,0.000579,8.60,0
3,01,2010-09-23 01:42:00,-78.443283,168.383636,17432.7,61.417,7.90,-9.91,192.625229,0.000581,8.94,0
4,01,2010-09-23 01:44:00,-78.453957,168.426804,17435.2,61.435,8.39,-10.14,192.624124,0.000580,9.03,0
...,...,...,...,...,...,...,...,...,...,...,...,...
595540,19,2010-11-23 09:50:00,-76.353119,-20.722179,17378.6,66.398,10.25,6.78,207.987745,0.000660,11.68,167
595541,19,2010-11-23 09:52:00,-76.345833,-20.675850,17366.3,66.525,10.08,6.80,207.950165,0.000660,11.62,167
595542,19,2010-11-23 09:54:00,-76.338570,-20.629745,17368.8,66.449,10.24,6.65,207.972627,0.000660,11.68,167
595543,19,2010-11-23 09:56:00,-76.331482,-20.582638,17357.0,66.633,10.34,6.53,207.918384,0.000660,11.57,167
